This file trains a simple encode / decoder architecture using PyTorch and exports it to ONNX format.
* First, the pytorch module is created. It consists of an encoder and a decoder, both implemented as simple feedforward neural networks.
Each of them simply has 3 strided convolutional layers (halfing the resolution in the encoder and increasing it in the decoder) followed by ReLU activations.
* Then, it is trained on a dataset of unlabelled images simply by reconstructing the input images from the latent representations.
* Finally, the trained model is exported to ONNX format for inference.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, out_channels=3):
        super().__init__()
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(32, out_channels, kernel_size=4, stride=2, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.deconv1(x))
        x = self.relu(self.deconv2(x))
        x = self.relu(self.deconv3(x))
        return x

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bitmind/caltech-101")

import torchvision.transforms as T

# Prepare transforms and dataloader
transform = T.Compose([
    T.ToTensor(),
    T.Resize((128, 128)),
])

def preprocess1(example):
    img = example["image"]
    return {"image": transform(img)}

train_ds = dataset["train"].map(preprocess1)

In [ ]:
from torch.utils.data import DataLoader

from tqdm import tqdm

train_ds.set_format(type="torch", columns=["image"])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# Instantiate models
encoder = Encoder()
decoder = Decoder()
autoencoder = nn.Sequential(encoder, decoder)
autoencoder.train()

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop for one epoch
for batch in tqdm(train_loader):
    imgs = batch["image"]
    optimizer.zero_grad()
    recon = autoencoder(imgs)
    loss = loss_fn(recon, imgs)
    loss.backward()
    optimizer.step()


In [ ]:
from matplotlib import pyplot as plt
from PIL import Image

# Load example image
img_path = "../../data/lantern.jpg"

image = Image.open(img_path)

plt.imshow(image)
plt.axis("off")
plt.show()


In [ ]:
image_tensor = transform(image).unsqueeze(0)
image_tensor.shape

with torch.no_grad():
    autoencoder.eval()
    encoder, decoder = autoencoder[0], autoencoder[1]

    latent = encoder(image_tensor)
    reconstructed = decoder(latent)
    
    reconstructed_img = reconstructed.squeeze(0).permute(1, 2, 0).numpy()

    plt.imshow(reconstructed_img)
    plt.axis("off")
    plt.show()


In [ ]:
# This first step is to export the encoder and decoder as simple ONNX files
# This however is not enough for klartraum engine to use as it needs 
# information about the tensor shapes

# Export the encoder
torch.onnx.export(encoder,
                  image_tensor,
                  "../../data/onnx/simple_encoder.onnx",
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'},
                                'output': {0: 'batch_size'}})

# Export the decoder
torch.onnx.export(decoder,
                  latent,
                  "../../data/onnx/simple_decoder.onnx",
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'},
                                'output': {0: 'batch_size'}})

In [ ]:
# this second step is to add shape information to the onnx files
# so that klartraum engine can use them

import onnx
from onnx import shape_inference

model = onnx.load("../../data/onnx/simple_encoder.onnx")
model = shape_inference.infer_shapes(model)
onnx.save(model, "../../data/onnx/simple_encoder_with_value_info.onnx")

model = onnx.load("../../data/onnx/simple_decoder.onnx")
model = shape_inference.infer_shapes(model)
onnx.save(model, "../../data/onnx/simple_decoder_with_value_info.onnx")


In [ ]:
# this third step is to add parameter information to the onnx files
# this would be enough for klartraum engine to use the onnx files


import onnx
from onnx import helper, TensorProto

def add_initializers_to_value_info(model: onnx.ModelProto, also_add_as_inputs=False):
    g = model.graph
    existing_vi = {vi.name for vi in g.value_info}
    existing_inputs = {vi.name for vi in g.input}
    init_names = {t.name for t in g.initializer}

    # Create ValueInfo for each initializer
    for t in g.initializer:
        if t.name not in existing_vi:
            vi = helper.make_tensor_value_info(
                name=t.name,
                elem_type=t.data_type,
                shape=list(t.dims) if t.dims else []
            )
            g.value_info.extend([vi])

    return model

# --- use it ---
model = onnx.load("../../data/onnx/simple_encoder_with_value_info.onnx")
model = add_initializers_to_value_info(model)
onnx.save(model, "../../data/onnx/simple_encoder_with_value_info_and_param_info.onnx")

model = onnx.load("../../data/onnx/simple_decoder_with_value_info.onnx")
model = add_initializers_to_value_info(model)
onnx.save(model, "../../data/onnx/simple_decoder_with_value_info_and_param_info.onnx")


In [ ]:
# Finally, for testing of the onnx implementation of klartraum engine
# we will freeze the intermediate values using onnxruntime
# and also export them as initializers in the onnx file

import onnx
import onnxruntime as ort
import numpy as np
from onnx import helper, TensorProto

def freeze_intermediate_values_with_onnx(model_path, input_data, output_path):
    """
    Run ONNX inference and store all intermediate tensor values as initializers.
    This version modifies the ONNX model to expose intermediate outputs first.
    
    Args:
        model_path: Path to the original ONNX model
        input_data: Input tensor (torch.Tensor or numpy array)
        output_path: Path to save the modified ONNX model
    """
    # Load the original model
    model = onnx.load(model_path)
    
    # Convert input to numpy if it's a torch tensor
    if hasattr(input_data, 'detach'):
        input_np = input_data.detach().cpu().numpy()
    else:
        input_np = input_data
    
    # Get all intermediate tensor names from the model
    intermediate_names = []
    output_names = {out.name for out in model.graph.output}
    
    for node in model.graph.node:
        for output_name in node.output:
            if output_name not in output_names:  # Skip final outputs
                intermediate_names.append(output_name)
    
    print(f"Found {len(intermediate_names)} intermediate tensors: {intermediate_names}")
    
    # Create a modified model with intermediate outputs exposed
    modified_model = onnx.load(model_path)
    graph = modified_model.graph
    
    # Add intermediate tensors as additional outputs
    for name in intermediate_names:
        # Try to infer the shape and type from the graph
        # For simplicity, we'll assume float32 and infer shape later
        try:
            intermediate_output = helper.make_tensor_value_info(
                name=name,
                elem_type=TensorProto.FLOAT,
                shape=None  # Empty shape, will be inferred
            )
            graph.output.append(intermediate_output)
        except Exception as e:
            print(f"Skipping intermediate tensor {name}: {e}")
            continue
    
    # Save the modified model temporarily
    temp_model_path = model_path.replace('.onnx', '_temp_with_intermediates.onnx')
    onnx.save(modified_model, temp_model_path)
    
    try:
        # Create inference session with the modified model
        sess = ort.InferenceSession(temp_model_path)
        input_name = sess.get_inputs()[0].name
        
        # Run inference and capture all values
        all_outputs = sess.run(intermediate_names, {input_name: input_np})
        
        print(f"Successfully captured {len(all_outputs)} tensor values")
        
        # Load the original model again for modification
        final_model = onnx.load(model_path)
        graph = final_model.graph
        added_count = 0
        
        # Get output names from the modified model
        modified_output_names = [out.name for out in modified_model.graph.output]
        
        for i, output_name in enumerate(modified_output_names):
            if output_name in intermediate_names and i < len(all_outputs):
                value = all_outputs[i]
                
                # Create TensorProto from numpy array
                tensor_name = output_name # f"{output_name}_frozen"
                
                # Handle different data types
                if value.dtype == np.float32:
                    data_type = TensorProto.FLOAT
                    vals = value.flatten().tolist()
                elif value.dtype == np.float64:
                    data_type = TensorProto.DOUBLE
                    vals = value.flatten().tolist()
                elif value.dtype == np.int64:
                    data_type = TensorProto.INT64
                    vals = value.flatten().astype(np.int64).tolist()
                elif value.dtype == np.int32:
                    data_type = TensorProto.INT32
                    vals = value.flatten().astype(np.int32).tolist()
                else:
                    print(f"Skipping tensor {output_name} with unsupported dtype: {value.dtype}")
                    continue
                
                tensor_proto = helper.make_tensor(
                    name=tensor_name,
                    data_type=data_type,
                    dims=list(value.shape),
                    vals=vals
                )
                
                # Add to initializers
                graph.initializer.append(tensor_proto)
                
                # Add ValueInfo for this tensor
                value_info = helper.make_tensor_value_info(
                    name=tensor_name,
                    elem_type=data_type,
                    shape=list(value.shape)
                )
                graph.value_info.append(value_info)
                
                print(f"Added {tensor_name} as initializer with shape {value.shape}")
                added_count += 1
        
        # Save the final model
        onnx.save(final_model, output_path)
        print(f"Successfully added {added_count} intermediate values as initializers")
        print(f"Saved model with frozen intermediates to: {output_path}")
        
        return final_model
        
    except Exception as e:
        print(f"Error running inference even with modified model: {e}")
        print("This might be due to shape inference issues. Trying simpler approach...")
        
        # Fallback: just run the original model and get what we can
        try:
            sess = ort.InferenceSession(model_path)
            input_name = sess.get_inputs()[0].name
            available_outputs = sess.run(None, {input_name: input_np})
            
            # Load original model for modification
            final_model = onnx.load(model_path)
            graph = final_model.graph
            
            # Add final output as frozen initializer
            output_names = [out.name for out in sess.get_outputs()]
            for i, output_name in enumerate(output_names):
                if i < len(available_outputs):
                    value = available_outputs[i]
                    tensor_name = output_name # f"{output_name}_frozen"
                    
                    tensor_proto = helper.make_tensor(
                        name=tensor_name,
                        data_type=TensorProto.FLOAT,
                        dims=list(value.shape),
                        vals=value.flatten().tolist()
                    )
                    
                    graph.initializer.append(tensor_proto)
                    
                    value_info = helper.make_tensor_value_info(
                        name=tensor_name,
                        elem_type=TensorProto.FLOAT,
                        shape=list(value.shape)
                    )
                    graph.value_info.append(value_info)
                    
                    print(f"Added {tensor_name} as initializer with shape {value.shape}")
            
            onnx.save(final_model, output_path)
            print("Saved model with final output frozen as fallback")
            return final_model
            
        except Exception as e2:
            print(f"Complete failure: {e2}")
            return None
    
    finally:
        # Clean up temporary file
        import os
        if os.path.exists(temp_model_path):
            os.remove(temp_model_path)

# Freeze intermediate values using ONNX Runtime directly
print("Freezing intermediate values for encoder using ONNX Runtime...")
try:
    encoder_with_frozen = freeze_intermediate_values_with_onnx(
        "../../data/onnx/simple_encoder_with_value_info_and_param_info.onnx",
        image_tensor,
        "../../data/onnx/simple_encoder_with_onnx_frozen_intermediates.onnx"
    )
except Exception as e:
    print(f"Error: {e}")
    print("Make sure onnxruntime is installed: pip install onnxruntime")

print("\nFreezing intermediate values for decoder using ONNX Runtime...")
try:
    decoder_with_frozen = freeze_intermediate_values_with_onnx(
        "../../data/onnx/simple_decoder_with_value_info_and_param_info.onnx",
        latent,
        "../../data/onnx/simple_decoder_with_onnx_frozen_intermediates.onnx"
    )
except Exception as e:
    print(f"Error: {e}")
    print("Make sure onnxruntime is installed: pip install onnxruntime")

print("\nDone! Created ONNX models with ONNX Runtime frozen intermediate values.")